# # 11__HALF__ — BERT Token Embeddings at 256 tokens, story points in (0, 100]
#
# The redo configuration requested after the main PS-3 evaluation: the label filter is relaxed
# from (0, 40] to the standard Deep-SE cap **(0, 100]** (drop SP ≤ 0 and SP > 100, Choetkiertikul
# et al. 2018) and the token window is raised from 128 to **256**, which cuts truncation from
# 29.5% of issues (41.8% of token content) to ~9% on this corpus. Everything else matches 03:
# frozen `bert-base-uncased`, last hidden state, float16 storage, per-project arrays.
#
# This is half `__HALF__` of the corpus (the 24.7 GB total does not fit one kernel output);
# `11b` carries the other half. `12_ps3_256_ablation` attaches both.
#
# **Input:** dataset `tawos-extracted` (`all_issues_sp.csv`). **Needs GPU + Internet ON.**
# **Outputs:** `emb256/<PROJECT>.npy` (float16 `[n, 256, 768]`), `meta/<PROJECT>.csv`
# (`issue_id, issue_key, storypoint, n_tokens` — untruncated length), `extraction_summary.csv`
# (per-project truncation at both 128 and 256), `run_metadata.json`.

In [ ]:
import glob
import json
import os
import re
import time

import numpy as np
import pandas as pd
import transformers

MAX_LEN = 256
BS = 64
MODEL = "bert-base-uncased"
HALF = "__HALF__"
PROJECTS = __PROJECTS_JSON__

csv = glob.glob("/kaggle/input/**/all_issues_sp.csv", recursive=True)[0]
df = pd.read_csv(csv)
df["storypoint"] = pd.to_numeric(df["storypoint"], errors="coerce")
df = df.dropna(subset=["storypoint"])
# the redo label filter: the standard Deep-SE cap, (0, 100]
df = df[(df.storypoint > 0) & (df.storypoint <= 100)]
df["text"] = df["text"].fillna("").astype(str)
df = df.reset_index(drop=True)
print(len(df), "issues |", df.project.nunique(), "projects | this half:", len(PROJECTS))
missing = set(PROJECTS) - set(df.project.unique())
assert not missing, f"projects missing from the corpus: {missing}"

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).cuda().eval()
torch.set_grad_enabled(False)

revision = None
try:
    from huggingface_hub import model_info
    revision = model_info(MODEL).sha
except Exception:
    pass

os.makedirs("/kaggle/working", exist_ok=True)
json.dump({
    "model": MODEL,
    "revision": revision,
    "max_length": MAX_LEN,
    "batch_size": BS,
    "dtype": "float16 autocast, fp16 storage",
    "label_filter": "(0, 100]",
    "half": HALF,
    "projects_in_half": PROJECTS,
    "issues": int(len(df)),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "numpy": np.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
}, open("/kaggle/working/run_metadata.json", "w"), indent=2)
print("gpu:", torch.cuda.get_device_name(0), "| half:", HALF, "| revision:", revision)

# ## Extraction loop
#
# Same two-pass discipline as 03: one untruncated pass records the true token length (so
# truncation statistics at 128 and 256 are both measurable), one padded pass fills the arrays.

In [ ]:
os.makedirs("/kaggle/working/emb256", exist_ok=True)
os.makedirs("/kaggle/working/meta", exist_ok=True)


def safe(n):
    return re.sub(r"[^A-Za-z0-9_]", "_", str(n))


summary = []
t0 = time.time()
for pi, proj in enumerate(PROJECTS):
    g = df[df.project == proj].reset_index(drop=True)
    out = np.empty((len(g), MAX_LEN, 768), dtype=np.float16)
    real_len = np.empty(len(g), dtype=np.int32)
    for b0 in range(0, len(g), BS):
        batch = list(g.text.values[b0:b0 + BS])
        real_len[b0:b0 + len(batch)] = [len(ids) for ids in
                                        tok(batch, add_special_tokens=True, truncation=False)["input_ids"]]
        enc = tok(batch, padding="max_length", truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            hs = model(**enc).last_hidden_state
        out[b0:b0 + len(batch)] = hs.half().cpu().numpy()
        del enc, hs

    key = safe(proj)
    np.save(f"/kaggle/working/emb256/{key}.npy", out)
    meta = g[["issue_id", "issue_key", "storypoint"]].copy()
    meta["n_tokens"] = real_len
    meta.to_csv(f"/kaggle/working/meta/{key}.csv", index=False)
    summary.append(dict(
        project=proj, file=f"{key}.npy", n=len(g), size_mb=round(out.nbytes / 1e6, 1),
        trunc_pct_128=round(float((real_len > 128).mean() * 100), 1),
        trunc_pct_256=round(float((real_len > 256).mean() * 100), 1),
        content_lost_at_128_pct=round(float(np.maximum(real_len - 128, 0).sum() / real_len.sum() * 100), 1),
        content_lost_at_256_pct=round(float(np.maximum(real_len - 256, 0).sum() / real_len.sum() * 100), 1),
    ))
    del out, real_len
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[{pi + 1}/{len(PROJECTS)}] {proj}: {len(g)} issues "
          f"| trunc@256 {summary[-1]['trunc_pct_256']}% | {time.time() - t0:.0f}s", flush=True)

sm = pd.DataFrame(summary)
sm.to_csv("/kaggle/working/extraction_summary.csv", index=False)
print("half", HALF, "DONE in", round((time.time() - t0) / 60, 1), "min")

# ## Verification
#
# Same checks as 03: shape, row count, finiteness and `issue_id` alignment for every array in
# this half, before the ~12 GB is published.

In [ ]:
total = 0
for _, r in sm.iterrows():
    a = np.load(f"/kaggle/working/emb256/{r.file}", mmap_mode="r")
    m = pd.read_csv(f"/kaggle/working/meta/{r.file[:-4]}.csv")
    g = df[df.project == r.project]
    assert a.shape == (len(m), MAX_LEN, 768), (r.file, a.shape, len(m))
    assert len(m) == len(g), r.file
    assert (m.issue_id.astype(str).values == g.issue_id.astype(str).values).all(), r.file
    assert np.isfinite(np.asarray(a[:4])).all(), r.file
    total += a.nbytes
print(f"all {len(sm)} project files verified | total {total / 1e9:.2f} GB")
w128 = float(np.average(sm.trunc_pct_128, weights=sm.n))
w256 = float(np.average(sm.trunc_pct_256, weights=sm.n))
c128 = float(np.average(sm.content_lost_at_128_pct, weights=sm.n))
c256 = float(np.average(sm.content_lost_at_256_pct, weights=sm.n))
print(f"this half, issue-weighted: trunc@128 {w128:.1f}% -> trunc@256 {w256:.1f}% "
      f"| content lost {c128:.1f}% -> {c256:.1f}%")